# Post-processing cyclones at 500 m points

This notebook applies the newer efficient methodology (KDTree crop + smooth multi-grid merge)
to cyclone partition outputs, preserving cyclone IDs.

It does **not** compute BMUs or GeoJSON statistics.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/lustre/geocean/WORK/users/montanoj/personal/Cyclones_NC')
GRID_NAMES = ['grid1', 'grid2', 'grid3', 'grid4']
POINTS_GEOJSON = PROJECT_ROOT / 'inputs' / 'isobath_10m_points_500m.geojson'

PARTITIONS_DIR = PROJECT_ROOT / 'outputs' / 'partitions_cyclones'
CROPPED_DIR = PROJECT_ROOT / 'outputs' / 'cropped_500m_partitions_cyclones_v2'
MERGED_DIR = PROJECT_ROOT / 'outputs' / 'merged_500m_partitions_cyclones_v2'

# Keep None for all IDs; or set, e.g., list(range(0, 100))
CYCLONE_IDS = None

# Keep None to auto-discover variables in PARTITIONS_DIR
VARIABLES = None

SMOOTH_STEEPNESS = 2.0
BLEND_BUFFER_KM = 30.0
TOLERANCE_DEG = 0.001
SKIP_EXISTING = True

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import postprocessing_cyclones_500m as pc

CROPPED_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_ROOT)
print('Partitions input:', PARTITIONS_DIR)
print('Cropped output:', CROPPED_DIR)
print('Merged output:', MERGED_DIR)
print('Cyclone filter:', CYCLONE_IDS if CYCLONE_IDS is not None else 'ALL')


## 1) Discover variables and available cyclone IDs


In [ ]:
variables = pc.discover_partition_variables(PARTITIONS_DIR, GRID_NAMES) if VARIABLES is None else VARIABLES
print('Variables:', variables)

for var in variables:
    ids = pc.available_cyclone_ids_for_var(PARTITIONS_DIR, var, GRID_NAMES, intersection=True)
    if ids:
        print(f'{var:8s} -> {len(ids)} common cyclone IDs across all grids (min={ids[0]}, max={ids[-1]})')
    else:
        print(f'{var:8s} -> no common cyclone IDs across all grids')


## 2) Crop per-cyclone files to 500 m reference points


In [ ]:
cropped_paths = pc.crop_partitions_to_points_geojson(
    project_root=PROJECT_ROOT,
    partitions_dir=PARTITIONS_DIR,
    output_dir=CROPPED_DIR,
    points_geojson_file=POINTS_GEOJSON,
    grid_names=GRID_NAMES,
    variables=variables,
    cyclone_ids=CYCLONE_IDS,
    point_match_tolerance=1e-6,
    skip_existing=SKIP_EXISTING,
    verbose=True,
)

print(f'\nCropped files: {len(cropped_paths)}')
if cropped_paths:
    print('First 10 files:')
    for p in sorted(cropped_paths)[:10]:
        print('  ', p.name)


## 3) Smooth-merge all grids by cyclone ID


In [ ]:
merged_paths = pc.merge_all_cyclone_partitions_smooth(
    cropped_dir=CROPPED_DIR,
    output_dir=MERGED_DIR,
    grid_names=GRID_NAMES,
    variables=variables,
    cyclone_ids=CYCLONE_IDS,
    steepness=SMOOTH_STEEPNESS,
    blend_buffer_km=BLEND_BUFFER_KM,
    tolerance_deg=TOLERANCE_DEG,
    skip_existing=SKIP_EXISTING,
    output_suffix='_merged_cyclones_500m',
)

print('\nMerged files:')
for var, path in merged_paths.items():
    blend = 'circular' if var in pc.CIRCULAR_BLEND_VARS else 'linear'
    size_mb = path.stat().st_size / 1e6 if path.is_file() else 0
    print(f'  {var:8s} {path.name} ({size_mb:.1f} MB, {blend} blend)')


## 4) Quick validation of one merged variable


In [ ]:
import xarray as xr

if merged_paths:
    sample_var = sorted(merged_paths.keys())[0]
    sample_file = merged_paths[sample_var]
    with xr.open_dataset(sample_file) as ds:
        print('Sample variable:', sample_var)
        print('Path:', sample_file)
        print('Dims:', dict(ds.sizes))
        print('Data vars:', list(ds.data_vars))
        if 'cyclone_id' in ds.coords:
            vals = ds['cyclone_id'].values
            print('cyclone_id count:', len(vals), 'min:', vals.min(), 'max:', vals.max())
else:
    print('No merged outputs found yet.')
